# Portfolio Rebalance

N stocks held at fixed target weights, reset on a calendar schedule.

In [1]:
from data import get_prices
from algorithm import PortfolioRebalance
from backtest import Backtester, build_report

In [2]:
from datetime import date

TARGET_WEIGHTS = {"AAPL": 0.4, "MSFT": 0.35, "GOOGL": 0.25}
BENCHMARK = "SPY"
START, END = "2018-01-01", date.today().isoformat()  # through today

prices = get_prices(list(TARGET_WEIGHTS), START, END)
benchmark = get_prices([BENCHMARK], START, END)[BENCHMARK]
prices.tail()

,AAPL,MSFT,GOOGL
Date,,,
2026-06-24,293.079987,365.459991,345.290009
2026-06-25,275.149994,352.829987,343.709991
2026-06-26,283.779999,372.970001,337.390015
2026-06-29,281.739990,368.570007,353.649994
2026-06-30,289.359985,373.019989,357.369995


In [3]:
algorithm = PortfolioRebalance(TARGET_WEIGHTS, frequency="QE")  # rebalance quarterly
result = Backtester().run(algorithm, prices, initial_capital=100_000, commission_bps=5, slippage_bps=5)
result.equity.tail()

Date
2026-06-24    658880.423718
2026-06-25    634033.500337
2026-06-26    651740.461293
2026-06-29    655027.751396
2026-06-30    666604.693683
dtype: float64

In [4]:
report = build_report(result, benchmark_prices=benchmark)
report.stats

{'total_return': np.float64(5.672719656490215),
 'cagr': np.float64(0.251239248433369),
 'annual_volatility': np.float64(0.26304474785293364),
 'sharpe': 0.9835122729989041,
 'sortino': 1.3320098931923667,
 'max_drawdown': -0.3421718656517936,
 'calmar': np.float64(0.7342487026359995),
 'win_rate': 0.5487347703842549,
 'trade_count': 1,
 'benchmark_total_return': np.float64(2.1648934261118344),
 'benchmark_cagr': np.float64(0.1457409961792573)}

**Reading the numbers:** `sharpe`/`sortino` above ~1 is solid, above ~2 is excellent (return per unit of risk taken). `max_drawdown` is the worst peak-to-trough loss you'd have sat through -- closer to 0 is better. Compare `cagr` to `benchmark_cagr` to see whether the strategy actually beat just holding the benchmark.

In [5]:
for fig in report.figures:
    fig.show()